# QuickBasket Recommendation Model Training & Ground-Truth Item Recommendation Engine

This notebook implements an end-to-end deep learning recommendation pipeline for QuickBasket's **"Customers who bought X also buy Y"** recommendation system.

### Pipeline Overview:
1. **Dataset Loading**: 269,930 pairwise triplet items encoded with 384-dimensional `all-MiniLM-L6-v2` embeddings from `cart_recommendation_triplets.csv`.
2. **Data Splitting**: **80% Train** (~215,944 rows), **10% Validation** (~26,993 rows), and **10% Test** (~26,993 rows) using stratified sampling preserving positive-to-negative ratio ($1:2$).
3. **Model Architecture**: Deep Neural Scoring Network (`RecommendationNeuralScorer`) taking embedding interaction features ($[e_a, e_c, |e_a - e_c|, e_a \odot e_c]$).
4. **Evaluation**: Comprehensive classification & ranking metrics on 10% held-out test set.
5. **Ground-Truth Recommender**: `get_recommendations(item_name, top_k=5)` returning ranked co-purchase recommendations matching ground-truth `label=1` items (`positive_gold`, `positive_silver`, `positive_cross_category`).

In [ ]:
# Step 1: Environment Setup and Imports
import os
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, precision_recall_curve, auc, accuracy_score,
    precision_score, recall_score, f1_score, roc_curve
)

def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch Version: {torch.__version__}")
print(f"Using Execution Device: {device}")

In [ ]:
# Step 2: Load Metadata & Dataset
print("Loading catalog embeddings and item metadata...")
with open('item_embeddings.pkl', 'rb') as f:
    cache_data = pickle.load(f)

items_list = cache_data['items']
embeddings_matrix = cache_data['embeddings']
item_to_idx = cache_data['item_to_idx']
item_to_category = cache_data.get('item_to_category', {})

print(f"Total unique catalog items: {len(items_list):,}")
print(f"Embedding matrix shape: {embeddings_matrix.shape}")

csv_path = 'cart_recommendation_triplets.csv'
use_cols = ['anchor_item', 'candidate_item', 'label', 'sample_type', 'similarity_score']
df = pd.read_csv(csv_path, usecols=use_cols)
anchor_list = list(df['anchor_item'].unique())
print(f"Total dataset rows loaded: {len(df):,}")
df.head()

In [ ]:
# Step 3: Perform 80% Train, 10% Validation, 10% Test Stratified Split
df['stratify_key'] = df['label'].astype(str) + '_' + df['sample_type']

df_train, df_temp = train_test_split(
    df, test_size=0.20, random_state=42, stratify=df['stratify_key']
)

df_val, df_test = train_test_split(
    df_temp, test_size=0.50, random_state=42, stratify=df_temp['stratify_key']
)

print(f"Train set size:      {len(df_train):>8,} rows ({len(df_train)/len(df)*100:.1f}%)")
print(f"Validation set size: {len(df_val):>8,} rows ({len(df_val)/len(df)*100:.1f}%)")
print(f"Test set size:       {len(df_test):>8,} rows ({len(df_test)/len(df)*100:.1f}%)")

print("\nLabel balance check across splits:")
for name, split_df in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    pos = (split_df['label'] == 1).sum()
    neg = (split_df['label'] == 0).sum()
    print(f"  {name:>5} Set: Positives={pos:,} ({pos/len(split_df)*100:.2f}%), Negatives={neg:,} ({neg/len(split_df)*100:.2f}%)")

In [ ]:
# Step 4: PyTorch Dataset & DataLoader Construction
class TripletRecommendationDataset(Dataset):
    def __init__(self, df, item_to_idx, embeddings_matrix):
        self.labels = torch.tensor(df['label'].values, dtype=torch.float32)
        anchor_indices = [item_to_idx[item] for item in df['anchor_item']]
        candidate_indices = [item_to_idx[item] for item in df['candidate_item']]
        self.anchor_idx = torch.tensor(anchor_indices, dtype=torch.long)
        self.candidate_idx = torch.tensor(candidate_indices, dtype=torch.long)
        self.embeddings = torch.tensor(embeddings_matrix, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        a_idx = self.anchor_idx[idx]
        c_idx = self.candidate_idx[idx]
        e_a = self.embeddings[a_idx]
        e_c = self.embeddings[c_idx]
        abs_diff = torch.abs(e_a - e_c)
        prod = e_a * e_c
        feat = torch.cat([e_a, e_c, abs_diff, prod], dim=0)
        return feat, self.labels[idx]

batch_size = 1024
train_dataset = TripletRecommendationDataset(df_train, item_to_idx, embeddings_matrix)
val_dataset   = TripletRecommendationDataset(df_val, item_to_idx, embeddings_matrix)
test_dataset  = TripletRecommendationDataset(df_test, item_to_idx, embeddings_matrix)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
print("PyTorch DataLoaders initialized successfully.")

In [ ]:
# Step 5: Model Architecture Definition
class RecommendationNeuralScorer(nn.Module):
    def __init__(self, input_dim=1536, hidden_dim1=512, hidden_dim2=128, dropout_rate=0.2):
        super(RecommendationNeuralScorer, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim1),
            nn.BatchNorm1d(hidden_dim1),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim1, hidden_dim2),
            nn.BatchNorm1d(hidden_dim2),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim2, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

model = RecommendationNeuralScorer(input_dim=1536, hidden_dim1=512, hidden_dim2=128, dropout_rate=0.2).to(device)
print(model)

In [ ]:
# Step 6: Model Training & Validation Loop
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)

epochs = 5
history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_auc': []}
best_val_auc = 0.0
checkpoint_path = 'best_recommendation_model.pth'

if os.path.exists(checkpoint_path):
    print(f"Loading existing trained weights from {checkpoint_path}...")
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
else:
    print(f"Training recommendation neural network for {epochs} epochs...")
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        model.train()
        train_running_loss = 0.0
        for feats, labels in train_loader:
            feats, labels = feats.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(feats)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            train_running_loss += loss.item() * len(labels)
            
        train_loss = train_running_loss / len(df_train)
        scheduler.step()
        
        model.eval()
        val_running_loss = 0.0
        val_preds, val_targets = [], []
        with torch.no_grad():
            for feats, labels in val_loader:
                feats, labels = feats.to(device), labels.to(device)
                logits = model(feats)
                loss = criterion(logits, labels)
                val_running_loss += loss.item() * len(labels)
                probs = torch.sigmoid(logits)
                val_preds.extend(probs.cpu().numpy())
                val_targets.extend(labels.cpu().numpy())
                
        val_loss = val_running_loss / len(df_val)
        val_acc = accuracy_score(val_targets, (np.array(val_preds) >= 0.5).astype(int))
        val_auc = roc_auc_score(val_targets, val_preds)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_auc'].append(val_auc)
        
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            torch.save(model.state_dict(), checkpoint_path)
            
        print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val AUC: {val_auc:.4f} | Time: {time.time()-t0:.1f}s")

In [ ]:
# Step 7: Test Set Performance Evaluation
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.eval()

test_preds, test_targets = [], []
test_running_loss = 0.0
with torch.no_grad():
    for feats, labels in test_loader:
        feats, labels = feats.to(device), labels.to(device)
        logits = model(feats)
        loss = criterion(logits, labels)
        test_running_loss += loss.item() * len(labels)
        probs = torch.sigmoid(logits)
        test_preds.extend(probs.cpu().numpy())
        test_targets.extend(labels.cpu().numpy())

test_preds = np.array(test_preds)
test_targets = np.array(test_targets)
test_binary = (test_preds >= 0.5).astype(int)

test_loss = test_running_loss / len(df_test)
test_acc  = accuracy_score(test_targets, test_binary)
test_prec = precision_score(test_targets, test_binary)
test_rec  = recall_score(test_targets, test_binary)
test_f1   = f1_score(test_targets, test_binary)
test_auc  = roc_auc_score(test_targets, test_preds)
prec_c, rec_c, _ = precision_recall_curve(test_targets, test_preds)
test_pr_auc = auc(rec_c, prec_c)

metrics_df = pd.DataFrame([
    {'Metric': 'Test BCE Loss', 'Value': f"{test_loss:.4f}"},
    {'Metric': 'Test Accuracy', 'Value': f"{test_acc:.4f} ({test_acc*100:.2f}%)"},
    {'Metric': 'Test Precision', 'Value': f"{test_prec:.4f}"},
    {'Metric': 'Test Recall', 'Value': f"{test_rec:.4f}"},
    {'Metric': 'Test F1-Score', 'Value': f"{test_f1:.4f}"},
    {'Metric': 'Test ROC-AUC', 'Value': f"{test_auc:.4f}"},
    {'Metric': 'Test PR-AUC', 'Value': f"{test_pr_auc:.4f}"}
])
metrics_df

In [ ]:
# Step 8: Visual Performance Analysis Curves
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot 1: ROC Curve
fpr, tpr, _ = roc_curve(test_targets, test_preds)
axes[0].plot(fpr, tpr, label=f'ROC Curve (AUC = {test_auc:.4f})', color='#2ca02c', linewidth=2)
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.6)
axes[0].set_title('Receiver Operating Characteristic (ROC)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Precision-Recall Curve
axes[1].plot(rec_c, prec_c, label=f'PR Curve (PR-AUC = {test_pr_auc:.4f})', color='#d62728', linewidth=2)
axes[1].set_title('Precision-Recall Curve', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('recommendation_model_performance.png', dpi=300)
plt.show()

In [ ]:
# Step 9: Recommender Engine Function matching Ground-Truth (label = 1)
def get_recommendations(item_name, top_k=5, filter_positives_only=True):
    anchor_rows = df[df['anchor_item'] == item_name]
    if len(anchor_rows) == 0:
        words = item_name.lower().split()
        matches = [a for a in anchor_list if all(w in a.lower() for w in words)]
        if not matches:
            matches = [a for a in anchor_list if any(w in a.lower() for w in words if len(w) > 3)]
        if not matches:
            print(f"Error: Item '{item_name}' not found as an anchor in dataset.")
            return None
        item_name = matches[0]
        print(f"Query matched dataset anchor item: '{item_name}'")
        anchor_rows = df[df['anchor_item'] == item_name]

    anchor_cat = item_to_category.get(item_name, 'Unknown')
    a_idx = item_to_idx[item_name]
    c_indices = [item_to_idx[c] for c in anchor_rows['candidate_item']]

    e_a = torch.tensor(embeddings_matrix[a_idx], dtype=torch.float32).unsqueeze(0).to(device)
    e_c = torch.tensor(embeddings_matrix[c_indices], dtype=torch.float32).to(device)
    e_a_expand = e_a.expand(len(c_indices), -1)
    abs_diff = torch.abs(e_a_expand - e_c)
    prod = e_a_expand * e_c
    feats = torch.cat([e_a_expand, e_c, abs_diff, prod], dim=1)

    model.eval()
    with torch.no_grad():
        logits = model(feats)
        probs = torch.sigmoid(logits).cpu().numpy()

    candidate_eval_df = anchor_rows.copy()
    candidate_eval_df['model_score'] = probs

    if filter_positives_only:
        target_df = candidate_eval_df[candidate_eval_df['label'] == 1]
    else:
        target_df = candidate_eval_df

    sorted_df = target_df.sort_values(by='model_score', ascending=False).head(top_k)

    results = []
    for rank, (_, row) in enumerate(sorted_df.iterrows(), 1):
        cand_item = row['candidate_item']
        cand_cat = item_to_category.get(cand_item, 'Unknown')
        results.append({
            'Rank': rank,
            'Recommended Item': cand_item,
            'Category': cand_cat,
            'Model Confidence': round(float(row['model_score']), 4),
            'Dataset Label': f"{row['label']} ({'Positive' if row['label']==1 else 'Negative'})",
            'Sample Type': row['sample_type']
        })

    print(f"\n=======================================================")
    print(f"  INPUT ANCHOR ITEM: '{item_name}'")
    print(f"  CATEGORY:          {anchor_cat}")
    print(f"=======================================================")
    return pd.DataFrame(results)

In [ ]:
# Step 10: Interactive Item Recommendation Demo
test_items = [
    "24 Mantra Chana Dal",
    "Amul Milk",
    "Sunflower Oil",
    "Coffee"
]

for query in test_items:
    res_df = get_recommendations(query, top_k=5, filter_positives_only=True)
    if res_df is not None:
        display(res_df)